In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Fixer les graines pour la reproductibilité
np.random.seed(42)
tf.random.set_seed(42)

# Génération des 600 points en deux classes entrelacées (lunes)
X, y = make_moons(n_samples=600, noise=0.35, random_state=42)
print("Forme de X :", X.shape, "| Forme de y :", y.shape)


#### **Cellule 2 : Découpage train/test et mise à l'échelle (sans fuite)**

# 1. Découpage stratifié (ex: 20% de test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 2. Mise à l'échelle ajustée uniquement sur le train
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


#### **Cellule 3 : Référence non neuronale (Régression Logistique)**

ref_model = LogisticRegression(random_state=42)
ref_model.fit(X_train_scaled, y_train)

ref_preds = ref_model.predict(X_test_scaled)
ref_acc = accuracy_score(y_test, ref_preds)
ref_f1 = f1_score(y_test, ref_preds, average='macro')
ref_cm = confusion_matrix(y_test, ref_preds)

print(f"Référence - Accuracy : {ref_acc:.3f} | F1 macro : {ref_f1:.3f}")
print("Matrice de confusion :\n", ref_cm)


#### **Cellule 4 : Entraînement et comparaison des réseaux de neurones (Multi-graines)**

def entrainer_evaluer_reseau(units, dropout_rate, lr, graines=[42, 100, 200]):
    accs, f1s = [], []
    for g in graines:
        tf.random.set_seed(g)
        np.random.seed(g)

        # Sous-découpage pour la validation (arrêt précoce)
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_scaled, y_train, test_size=0.2, stratify=y_train, random_state=g
        )

        model = Sequential([
            Dense(units, activation='relu', input_shape=(2,)),
            Dropout(dropout_rate),
            Dense(1, activation='sigmoid')
        ])

        optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
        model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

        early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

        model.fit(X_tr, y_tr, validation_data=(X_val, y_val),
                  epochs=100, batch_size=16, callbacks=[early_stopping], verbose=0)

        preds_proba = model.predict(X_test_scaled, verbose=0)
        preds = (preds_proba > 0.5).astype(int).flatten()

        accs.append(accuracy_score(y_test, preds))
        f1s.append(f1_score(y_test, preds, average='macro'))

    return np.mean(accs), np.std(accs), np.mean(f1s)

# Configuration A : Couche cachée modérée, pas de dropout
acc_a_m, acc_a_s, f1_a_m = entrainer_evaluer_reseau(units=8, dropout_rate=0.0, lr=0.01)

# Configuration B : Couche cachée plus large + Dropout
acc_b_m, acc_b_s, f1_b_m = entrainer_evaluer_reseau(units=16, dropout_rate=0.2, lr=0.01)

print(f"Config A (8 neurones) - Accuracy moy : {acc_a_m:.3f} ± {acc_a_s:.3f}")
print(f"Config B (16 neurones + Dropout) - Accuracy moy : {acc_b_m:.3f} ± {acc_b_s:.3f}")


## Taille du jeu de test
n_test = len(y_test)
p_ref = ref_acc

# Formule de la marge de bruit à 95% (1.96 * sqrt(p*(1-p)/n))
marge_bruit = 1.96 * np.sqrt((p_ref * (1 - p_ref)) / n_test)

print(f"Taille du test : {n_test}")
print(f"Précision de référence : {p_ref:.3f}")
print(f"Marge de bruit statistique (95%) : ± {marge_bruit:.3f}")




Forme de X : (600, 2) | Forme de y : (600,)
Référence - Accuracy : 0.850 | F1 macro : 0.850
Matrice de confusion :
 [[50 10]
 [ 8 52]]


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr

Config A (8 neurones) - Accuracy moy : 0.867 ± 0.030
Config B (16 neurones + Dropout) - Accuracy moy : 0.894 ± 0.010
Taille du test : 120
Précision de référence : 0.850
Marge de bruit statistique (95%) : ± 0.064


## ✍️ Conclusion du Défi : Réseau de neurones vs Méthode simple

### 1. Performance comparative
Sur ce problème non linéaire, la régression logistique (référence) atteint une précision de **0.850**. Le réseau de neurones (configurations à 8 et 16 neurones) obtient quant à lui une précision moyenne de **0.889**. Le réseau affiche donc une légère amélioration moyenne de $+0.039$.

### 2. Analyse statistique et bruit
Avec un jeu de test de $n = 120$ échantillons (20% de 600), la marge de bruit statistique calculée à 95% s'élève à **$\pm 0.064$**.

### 3. Réponse finale au client
**Peut-on garantir que le réseau fait mieux ? Strictement parlant, non.**
Bien que le réseau de neurones surperforme légèrement la méthode simple en moyenne, l'écart de performance ($+0.039$) est inférieur à la marge de bruit statistique ($\pm 0.064$). Cet écart ne permet donc pas d'écarter totalement l'hypothèse d'une variation due au hasard de l'échantillonnage. Pour obtenir une garantie formelle, il faudrait augmenter la taille du jeu de données ou du test.